In [6]:
import geopandas as gpd
import requests
from io import BytesIO

WFS_URL = "https://openmaps.gov.bc.ca/geo/pub/WHSE_LEGAL_ADMIN_BOUNDARIES.DRP_MOF_FIRE_ZONES_SP/ows"

params = {
    "service": "WFS",
    "version": "2.0.0",
    "request": "GetFeature",
    "typeNames": "WHSE_LEGAL_ADMIN_BOUNDARIES.DRP_MOF_FIRE_ZONES_SP",
    "outputFormat": "application/json",
    "CQL_FILTER": "MOF_FIRE_CENTRE_NAME='Cariboo Fire Centre'",
}

response = requests.get(WFS_URL, params=params, timeout=120)
response.raise_for_status()

zones_gdf = gpd.read_file(BytesIO(response.content))

cariboo_zones_gdf = zones_gdf[["MOF_FIRE_ZONE_NAME", "geometry"]].copy()
cariboo_zones_gdf = cariboo_zones_gdf.rename(columns={"MOF_FIRE_ZONE_NAME": "Zone_Name"})




print(cariboo_zones_gdf[["Zone_Name"]])
print(cariboo_zones_gdf.crs)


                   Zone_Name
0   100 Mile House Fire Zone
1        Chilcotin Fire Zone
2          Quesnel Fire Zone
3  Central Cariboo Fire Zone
EPSG:3005


In [7]:
cariboo_zones_gdf.columns

Index(['Zone_Name', 'geometry'], dtype='object')

In [9]:
import os
OUT_DIR = "../../extracted/zone_data"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
geojson_path = os.path.join(OUT_DIR, "cariboo_zones.geojson")
cariboo_zones_gdf.to_crs(epsg=4326).to_file(geojson_path, driver="GeoJSON")